In [1]:
from google.colab import userdata, drive

GOOGLE_API_KEY   = userdata.get('gemini')
PINECONE_API_KEY = userdata.get('pinecone')

drive.mount('/content/drive', force_remount=True)

DATA_DIR = "/content/drive/MyDrive/data"

Mounted at /content/drive


In [2]:
pip install langchain_google_genai

In [3]:
pip install pinecone

In [4]:
import os
import re
import math
import time
import hashlib
from dataclasses import dataclass, field
from collections import defaultdict

import google.generativeai as genai
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [5]:
INDEX_NAME      = "my-rag-index"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"   # 384-dim, stronger retrieval than paraphrase variant
EMBED_DIM       = 384
CHUNK_SIZE      = 5                     # sentences per chunk window
CHUNK_STEP      = 4                     # step size → 1-sentence overlap between chunks
UPSERT_BATCH    = 64                    # vectors per Pinecone batch call
TOP_K_RETRIEVE  = 5                     # chunks passed to Gemini

genai.configure(api_key=GOOGLE_API_KEY)

In [6]:
@dataclass
class Chunk:
    """
    A single unit of text that will become one vector in the database.
    Using a dataclass here (instead of a plain dict) gives us:
      - autocomplete in IDEs
      - type hints
      - cleaner __repr__ for debugging
    """
    chunk_id:   str             # unique ID derived from content hash
    text:       str             # the actual chunk text
    source:     str             # original filename
    position:   int             # chunk index within the file
    doc_type:   str             # inferred category (policy / faq / guide / text)
    keywords:   list[str]       # TF-IDF top keywords
    word_count: int             # number of words


In [7]:
def split_into_sentences(text: str) -> list[str]:
    """
    Split raw text into individual sentences.

    We use a simple regex that breaks on . ! ? followed by whitespace
    and a capital letter. This avoids splitting on "Mr. Smith" etc.
    """
    sentence_end = re.compile(r'(?<=[.!?])\s+(?=[A-Z])')
    sentences = sentence_end.split(text.strip())
    return [s.strip() for s in sentences if s.strip()]


def sliding_window_chunks(sentences: list[str], window: int, step: int) -> list[str]:
    """
    Group sentences into overlapping windows.

    CONCEPT — sliding window:
    Suppose window=5, step=4. We take sentences [0-4], then [4-8], then [8-12].
    The overlap (sentence 4, sentence 8, ...) is the "bridge" sentence
    that appears in two consecutive chunks so no idea is ever cut off entirely.

    This is more principled than a fixed character overlap because we always
    overlap at a clean sentence boundary, never mid-word.

       [s0  s1  s2  s3  s4]
                       [s4  s5  s6  s7  s8]
                                       [s8  s9  ...]
    """
    chunks = []
    total = len(sentences)
    start = 0
    while start < total:
        end = min(start + window, total)
        chunk_text = " ".join(sentences[start:end])
        chunks.append(chunk_text)
        if end == total:
            break
        start += step
    return chunks


def infer_doc_type(filename: str) -> str:
    """Categorise a file by its name so we can filter by type later."""
    name = filename.lower()
    if "security" in name:       return "security"
    if "training" in name:    return "training"
    if "travel" in name or "fuel" in name  : return "travel"
    if "policies" in name or "conduct" in name or "performance" in name:    return "work policies"
    return "text"


def make_chunk_id(source: str, position: int, text: str) -> str:
    """
    Stable, unique ID for a chunk.
    Using an MD5 hash of the content means re-running the pipeline
    on the same file produces the same IDs → safe to re-upsert (idempotent).
    """
    raw = f"{source}::{position}::{text[:80]}"
    return hashlib.md5(raw.encode()).hexdigest()

In [8]:
STOPWORDS = {
    "the","is","are","was","were","a","an","and","or","of","to","in","for",
    "on","with","as","by","at","from","that","this","it","be","has","have",
    "had","will","would","can","could","should","may","might","not","but",
    "if","then","than","so","such","into","their","there","about","over",
    "under","between","during","using","used","also","which","when","who",
    "how","what","where","its","our","your","my","any","all","each","per",
}


def tokenise(text: str) -> list[str]:
    words = re.findall(r"[a-zA-Z]{3,}", text.lower())
    return [w for w in words if w not in STOPWORDS]


def build_tfidf_keywords(chunks: list["Chunk"], top_k: int = 8) -> list[list[str]]:
    """
    TF-IDF keyword extraction across the full corpus of chunks.

    WHY TF-IDF instead of raw frequency (what the instructor did):
      Raw frequency picks "loan" as a keyword in every single finance chunk —
      it's common across ALL docs so it tells us nothing unique about THIS chunk.
      TF-IDF down-weights terms that appear everywhere and boosts terms that are
      distinctive to this specific chunk.

      TF  = term frequency within THIS chunk
      IDF = log(total_chunks / chunks_containing_this_term)
      Score = TF × IDF → high for rare-but-local terms

    This gives much more meaningful metadata for filtering.
    """
    N = len(chunks)
    if N == 0:
        return []

    # Count how many chunks each term appears in (for IDF)
    doc_freq: dict[str, int] = defaultdict(int)
    chunk_tokens: list[list[str]] = []

    for chunk in chunks:
        tokens = tokenise(chunk.text)
        unique_in_chunk = set(tokens)
        chunk_tokens.append(tokens)
        for term in unique_in_chunk:
            doc_freq[term] += 1

    all_keywords = []
    for tokens in chunk_tokens:
        if not tokens:
            all_keywords.append([])
            continue

        # Term frequency for this chunk
        tf: dict[str, float] = defaultdict(float)
        for t in tokens:
            tf[t] += 1.0 / len(tokens)

        # TF-IDF score
        scores = {
            term: freq * math.log((N + 1) / (doc_freq[term] + 1))
            for term, freq in tf.items()
        }

        top = sorted(scores, key=scores.get, reverse=True)[:top_k]
        all_keywords.append(top)

    return all_keywords

In [9]:
def load_and_chunk_docs(data_dir: str) -> list[Chunk]:
    """
    Read every .txt file in data_dir, sentence-split it,
    apply sliding-window chunking, and return a list of Chunk objects.
    """
    raw_chunks: list[Chunk] = []

    for filename in sorted(os.listdir(data_dir)):
        if not filename.endswith(".txt"):
            continue

        filepath = os.path.join(data_dir, filename)
        with open(filepath, "r", encoding="utf-8") as f:
            content = f.read()

        sentences  = split_into_sentences(content)
        texts      = sliding_window_chunks(sentences, window=CHUNK_SIZE, step=CHUNK_STEP)
        doc_type   = infer_doc_type(filename)

        for i, text in enumerate(texts):
            chunk = Chunk(
                chunk_id   = make_chunk_id(filename, i, text),
                text       = text,
                source     = filename,
                position   = i,
                doc_type   = doc_type,
                keywords   = [],        # filled in after TF-IDF pass
                word_count = len(text.split()),
            )
            raw_chunks.append(chunk)

        print(f"  {filename:40s} → {len(texts)} chunks")

    return raw_chunks


print("=== Loading and chunking documents ===")
chunks = load_and_chunk_docs(DATA_DIR)
print(f"\nTotal chunks: {len(chunks)}")

=== Loading and chunking documents ===
  Fuel and Mileage Policy.txt              → 11 chunks
  International Travel.txt                 → 13 chunks
  Travel Policy.txt                        → 19 chunks
  code of conduct.txt                      → 11 chunks
  it security and data privacy.txt         → 12 chunks
  learning and tuition.txt                 → 8 chunks
  leave_and_absence.txt                    → 13 chunks
  performance and compensation.txt         → 9 chunks

Total chunks: 96


In [10]:
print("\n=== Computing TF-IDF keywords ===")
all_keywords = build_tfidf_keywords(chunks, top_k=8)
for chunk, kws in zip(chunks, all_keywords):
    chunk.keywords = kws

# Quick sanity check — print first chunk
print("\nSample chunk:")
print(f"  source   : {chunks[0].source}")
print(f"  doc_type : {chunks[0].doc_type}")
print(f"  keywords : {chunks[0].keywords}")
print(f"  text     : {chunks[0].text[:120]}...")



=== Computing TF-IDF keywords ===

Sample chunk:
  source   : Fuel and Mileage Policy.txt
  doc_type : travel
  keywords : ['vehicle', 'usage', 'mileage', 'rate', 'personal', 'operate', 'guiding', 'principles']
  text     : # Corporate Travel Policy: Personal Vehicle, Fuel, and Mileage Reimbursement
**Document ID:** TRV-POL-3012-V2
**Effectiv...


In [11]:
print("\n=== Embedding chunks ===")
print(f"Model: {EMBEDDING_MODEL}")

embedder   = SentenceTransformer(EMBEDDING_MODEL)
texts      = [c.text for c in chunks]
embeddings = embedder.encode(texts, show_progress_bar=True, batch_size=64)

print(f"Embedding shape: {embeddings.shape}")   # (n_chunks, 384)


=== Embedding chunks ===
Model: all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embedding shape: (96, 384)


In [12]:
print("\n=== Setting up Pinecone ===")

pc = Pinecone(api_key=PINECONE_API_KEY)

existing_indexes = [idx.name for idx in pc.list_indexes()]

if INDEX_NAME not in existing_indexes:
    print(f"Creating index '{INDEX_NAME}'...")
    pc.create_index(
        name   = INDEX_NAME,
        dimension = EMBED_DIM,
        metric = "cosine",
        spec   = ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    # Poll until ready
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        print("  waiting for index to be ready...")
        time.sleep(5)
    print("  Index ready.")
else:
    print(f"Index '{INDEX_NAME}' already exists — connecting.")

index = pc.Index(INDEX_NAME)


=== Setting up Pinecone ===
Index 'my-rag-index' already exists — connecting.


In [13]:

def chunk_to_vector_record(chunk: Chunk, embedding: list[float]) -> dict:
    """
    Convert a Chunk + its embedding into the dict format Pinecone expects.

    Metadata is kept flat (no nested dicts) because Pinecone only supports
    flat key-value metadata for filtering.
    """
    return {
        "id": chunk.chunk_id,
        "values": embedding.tolist(),
        "metadata": {
            "text":       chunk.text,
            "source":     chunk.source,
            "position":   chunk.position,
            "doc_type":   chunk.doc_type,
            "keywords":   chunk.keywords,   # list[str] — Pinecone stores this as array
            "word_count": chunk.word_count,
        },
    }



In [14]:
print("\n=== Upserting vectors to Pinecone ===")

records = [chunk_to_vector_record(c, e) for c, e in zip(chunks, embeddings)]

total   = len(records)
batches = range(0, total, UPSERT_BATCH)

for batch_start in batches:
    batch = records[batch_start : batch_start + UPSERT_BATCH]
    index.upsert(vectors=batch)
    done = min(batch_start + UPSERT_BATCH, total)
    print(f"  Upserted {done}/{total} vectors")

print(f"\nAll {total} vectors in Pinecone.")
print(index.describe_index_stats())


=== Upserting vectors to Pinecone ===
  Upserted 64/96 vectors
  Upserted 96/96 vectors

All 96 vectors in Pinecone.
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '183',
                                    'content-type': 'application/json',
                                    'date': 'Sun, 03 May 2026 10:17:04 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '41',
                                    'x-pinecone-request-latency-ms': '40',
                                    'x-pinecone-response-duration-ms': '42'}},
 'dimension': 384,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 96}},
 'storageFullness': 0.0,
 'total_vector_count': 96,
 'vector_type': 'dense'}


In [27]:
from langchain_google_genai import ChatGoogleGenerativeAI
_gemini_model = ChatGoogleGenerativeAI(model = 'gemini-2.5-flash', api_key=GOOGLE_API_KEY)

def call_gemini(prompt: str, temperature: float = 0.2) -> str:
    """
    Thin wrapper around Gemini so we don't repeat boilerplate everywhere.
    temperature=0.2 keeps outputs focused and deterministic enough for
    query expansion / routing tasks, while still allowing natural phrasing.
    """
    response = _gemini_model.invoke(prompt)
    return response.text.strip()

In [28]:
def expand_query(user_query: str) -> list[str]:

    prompt = f"""You are helping improve a search query for a document retrieval system.
Rewrite the following user question in 3 different ways.
Use formal, document-like language (the kind found in policy documents or FAQs).
Each rewrite should use DIFFERENT vocabulary than the others.
Return ONLY a Python list of 3 strings, no explanation, no markdown.

Original question: "{user_query}"

Output format: ["rewrite 1", "rewrite 2", "rewrite 3"]"""

    raw = call_gemini(prompt, temperature=0.4)

    # Safely parse the list Gemini returns
    try:
        import ast
        rewrites = ast.literal_eval(raw)
        if isinstance(rewrites, list):
            return [user_query] + [str(r) for r in rewrites[:3]]
    except Exception:
        pass

    # Fallback: if parsing fails, just use the original query
    print("  [warn] query expansion parse failed — using original query only")
    return [user_query]


# ── 14. Step 1b — HyDE (Hypothetical Document Embeddings) ─────────────────────

def generate_hyde_passage(user_query: str) -> str:

    prompt = f"""Write a short formal paragraph (3-4 sentences) that directly answers
the following question, as if it were an excerpt from an official policy or FAQ document.
Use formal, third-person language. Invent plausible but generic details.
Do NOT say you don't know. Do NOT mention that this is hypothetical.

Question: "{user_query}"

Policy excerpt:"""

    return call_gemini(prompt, temperature=0.1)

In [29]:
def detect_doc_type_intent(user_query: str) -> str | None:

    prompt = f"""Classify the following question into ONE document category.
Choose from: policy, faq, guide, report, none
- policy  : rules, eligibility, terms, procedures
- security     : IT security related questions and rules
- training   : step-by-step instructions, walkthroughs, learning
- travel  : travel-related questions and rules, distance and fuel laws
- none    : the question could belong to multiple categories

Question: "{user_query}"

Respond with a single word only."""

    result = call_gemini(prompt).lower().strip()
    valid = {"policy", "faq", "guide", "report"}
    return result if result in valid else None


In [30]:
BROAD_TOP_K = 25   # retrieve wide — the reranker will prune this down later

def retrieve_for_query(
    query_text: str,
    top_k: int = BROAD_TOP_K,
    doc_type_filter: str | None = None,
) -> list[dict]:
    """
    Embed one query text and retrieve top_k chunks from Pinecone.
    Applies doc_type pre-filter if provided.
    """
    vec = embedder.encode([query_text])[0].tolist()

    pinecone_filter = (
        {"doc_type": {"$eq": doc_type_filter}} if doc_type_filter else None
    )

    results = index.query(
        vector           = vec,
        top_k            = top_k,
        include_metadata = True,
        filter           = pinecone_filter,
    )
    return results["matches"]


def pool_unique_matches(all_match_lists: list[list[dict]]) -> list[dict]:
    """
    Merge results from multiple queries (expansion + HyDE) into one
    deduplicated list. We keep the HIGHEST score seen for each chunk ID.

    WHY: if chunk X was retrieved by both the original query and a HyDE passage,
    it's a strong signal of relevance. Deduplication + max-score keeps it once.
    """
    best: dict[str, dict] = {}
    for matches in all_match_lists:
        for m in matches:
            cid = m["id"]
            if cid not in best or m["score"] > best[cid]["score"]:
                best[cid] = m
    return list(best.values())


In [31]:
def post_filter_keep_latest(matches: list[dict]) -> list[dict]:

    # Group by "base name" = everything before _v<N> or _<year> suffixes
    version_pattern = re.compile(r'(_v\d+|_20\d\d)', re.IGNORECASE)

    groups: dict[str, list[dict]] = defaultdict(list)
    for m in matches:
        source = m["metadata"].get("source", "")
        base   = version_pattern.sub("", source).lower()
        groups[base].append(m)

    filtered = []
    for base, group_matches in groups.items():
        # Sort filenames descending — latest version/year sorts last alphabetically
        group_matches.sort(key=lambda m: m["metadata"].get("source", ""), reverse=True)
        latest_source = group_matches[0]["metadata"].get("source", "")

        # Keep only chunks from the latest version of this document
        kept = [m for m in group_matches if m["metadata"].get("source") == latest_source]
        filtered.extend(kept)

    return filtered


In [32]:
from sentence_transformers import CrossEncoder

# We load this once at module level so it's not reloaded on every query.
# cross-encoder/ms-marco-MiniLM-L-6-v2 is a lightweight but strong reranker
# trained specifically to score (query, passage) relevance from 0 → 1.
print("\n=== Loading cross-encoder reranker ===")
_reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("  Reranker ready.")

FINAL_TOP_N = 5   # chunks we actually pass to Gemini


def rerank(user_query: str, matches: list[dict], top_n: int = FINAL_TOP_N) -> list[dict]:


    if not matches:
        return []

    # Build (query, passage) pairs — one per candidate chunk
    pairs = [
        (user_query, m["metadata"].get("text", ""))
        for m in matches
    ]

    # Score all pairs in one batch
    scores = _reranker.predict(pairs)

    # Attach the rerank score to each match dict
    for match, score in zip(matches, scores):
        match["rerank_score"] = float(score)

    # Sort by rerank score descending, keep top N
    reranked = sorted(matches, key=lambda m: m["rerank_score"], reverse=True)
    return reranked[:top_n]



=== Loading cross-encoder reranker ===


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Reranker ready.


In [33]:
def build_context(matches: list[dict]) -> str:
    """
    Format the final top-N chunks into a labelled context block.
    Each chunk is tagged with its source, its bi-encoder similarity score,
    and its cross-encoder rerank score so the output is fully traceable.
    """
    parts = []
    for i, m in enumerate(matches, 1):
        source       = m["metadata"].get("source", "unknown")
        sim_score    = m.get("score", 0.0)
        rerank_score = m.get("rerank_score", "n/a")
        text         = m["metadata"].get("text", "")

        header = f"[{i}] {source}  |  similarity={sim_score:.3f}  |  rerank={rerank_score:.3f}"
        parts.append(f"{header}\n{text}")

    return "\n\n---\n\n".join(parts)


In [34]:
def answer(question: str) -> str:
    """
    The complete advanced RAG pipeline.

    Flow:
      1. Expand the user's query into 3 variants            (multi-query)
      2. Generate a hypothetical document passage            (HyDE)
      3. Detect which doc_type the question targets          (LLM router)
      4. Embed all 5 texts, retrieve Top 25 each from Pinecone (pre-filtered)
      5. Pool + deduplicate all retrieved chunks
      6. Drop chunks from stale document versions            (post-filter)
      7. Cross-encoder re-scores survivors, keep Top 5       (reranker)
      8. Pass Top 5 + question to Gemini for grounded answer

    The user only calls this one function. All the complexity is internal.
    """

    print(f"\n{'='*60}")
    print(f"QUESTION: {question}")
    print('='*60)

    # ── Step 1: Query transformation ─────────────────────────────
    print("\n[1/5] Expanding query (multi-query + HyDE)...")

    query_variants = expand_query(question)
    print(f"  Variants: {len(query_variants)} (original + {len(query_variants)-1} expansions)")
    for v in query_variants:
        print(f"    • {v}")

    hyde_passage = generate_hyde_passage(question)
    print(f"  HyDE passage (first 100 chars): {hyde_passage[:100]}...")

    # All texts we'll embed and search with
    search_texts = query_variants + [hyde_passage]

    # ── Step 2a: Intent → pre-filter ─────────────────────────────
    print("\n[2/5] Detecting intent for pre-filter...")
    inferred_type = detect_doc_type_intent(question)
    print(f"  doc_type filter: {inferred_type or 'none (searching all types)'}")

    # ── Step 2b: Broad retrieval across all search texts ─────────
    print(f"\n[3/5] Retrieving Top {BROAD_TOP_K} chunks per query text...")
    all_match_lists = []
    for i, text in enumerate(search_texts, 1):
        label = "HyDE" if i == len(search_texts) else f"query variant {i}"
        matches = retrieve_for_query(text, top_k=BROAD_TOP_K, doc_type_filter=inferred_type)
        all_match_lists.append(matches)
        print(f"  [{label}] → {len(matches)} chunks")

    pooled = pool_unique_matches(all_match_lists)
    print(f"  Pooled unique chunks: {len(pooled)}")

    # ── Step 2c: Post-filter — drop stale versions ────────────────
    print("\n[4/5] Post-filtering (removing stale document versions)...")
    filtered = post_filter_keep_latest(pooled)
    print(f"  Chunks after post-filter: {len(filtered)}")

    # Safety check: if we filtered down to nothing, fall back to pooled
    if not filtered:
        print("  [warn] Post-filter removed everything — using pooled results.")
        filtered = pooled

    # ── Step 3: Rerank to Top 5 ───────────────────────────────────
    print(f"\n[5/5] Reranking {len(filtered)} chunks → Top {FINAL_TOP_N}...")
    top_chunks = rerank(question, filtered, top_n=FINAL_TOP_N)

    print("  Final chunks passed to Gemini:")
    for i, c in enumerate(top_chunks, 1):
        src   = c["metadata"].get("source", "?")
        rscore = c.get("rerank_score", 0)
        print(f"    [{i}] {src:35s}  rerank_score={rscore:.4f}")

    # ── Generate answer with Gemini ───────────────────────────────
    context = build_context(top_chunks)

    prompt = f"""You are a helpful assistant for a financial services company.
Answer the question below using ONLY the document excerpts provided.
Be specific. If the answer is not present in the excerpts, say:
"I couldn't find that in the available documents."
Do NOT use any knowledge from outside the excerpts.

=== Document Excerpts ===
{context}

=== Question ===
{question}

=== Answer ==="""

    final_answer = call_gemini(prompt, temperature=0.1)

    print(f"\nANSWER:\n{final_answer}")
    print('='*60)
    return final_answer


In [37]:

answer("what is the training period?")

##answer("What documents are needed for a loan against property?")

##answer("What is the EMI calculation formula?")

##answer("What's the allowance for missed EMI payments?")


QUESTION: what is the training period?

[1/5] Expanding query (multi-query + HyDE)...
  Variants: 4 (original + 3 expansions)
    • what is the training period?
    • What is the prescribed duration of the educational program?
    • Kindly specify the allocated timeframe for professional development.
    • Please indicate the standard interval designated for instructional activities.
  HyDE passage (first 100 chars): The standard training period for all newly onboarded personnel is a mandatory two (2) weeks, compris...

[2/5] Detecting intent for pre-filter...
  doc_type filter: none (searching all types)

[3/5] Retrieving Top 25 chunks per query text...
  [query variant 1] → 25 chunks
  [query variant 2] → 25 chunks
  [query variant 3] → 25 chunks
  [query variant 4] → 25 chunks
  [HyDE] → 25 chunks
  Pooled unique chunks: 46

[4/5] Post-filtering (removing stale document versions)...
  Chunks after post-filter: 46

[5/5] Reranking 46 chunks → Top 5...
  Final chunks passed to Gemini

'New hires must complete specific onboarding modules (HR-101, SEC-100, FIN-201) within their first thirty (30) calendar days of employment. All employees must also complete a 60-minute "Security and Compliance Refresher" annually by October 31st.'